In [1]:
from pathlib import Path
import json
import yaml

cwd = Path.cwd()
CONFIG_PATH = cwd.parent / "config" / "config.yaml"

with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

FILTERED_DIR = Path(cfg["paths"]["htem_filtered_data_root"]).resolve()
assert FILTERED_DIR.exists(), f"Filtered path does not exist: {FILTERED_DIR}"

XRD_KEY = "xrd_angle"

lengths = set()
bad_files = []

for lib_dir in FILTERED_DIR.iterdir():
    if not lib_dir.is_dir():
        continue

    samples_dir = lib_dir / "samples"
    if not samples_dir.exists():
        continue

    for sf in samples_dir.glob("sample *.json"):
        try:
            with sf.open("r", encoding="utf-8") as f:
                sample = json.load(f)
            xrd = sample.get(XRD_KEY, None)
            if not isinstance(xrd, list):
                bad_files.append(sf)
                continue
            lengths.add(len(xrd))
        except Exception:
            bad_files.append(sf)

print("Unique xrd_angle lengths in Filtered set:", lengths)
print("Bad / unreadable files:", len(bad_files))

assert len(lengths) == 1, "ERROR: Multiple xrd_angle lengths still present"
assert len(bad_files) == 0, "ERROR: Invalid sample files detected"

print("Filtered dataset is structurally clean.")

Unique xrd_angle lengths in Filtered set: {661}
Bad / unreadable files: 0
Filtered dataset is structurally clean.
